# 인공지능 기초 67~72쪽 데이터 전처리
위에서부터 셀을 차례로 실행해 보세요. (셀을 누르고 Shift + Enter)

교과서는 코랩에 파일을 올려서 쓰지만, 이 노트북은 인터넷 주소에서 Iris1.csv와 Iris2.csv를 바로 읽도록 바꿔 두었어요.

파일 업로드하기

교과서 코드(파일 업로드)는 아래와 같아요. 이 노트북에서는 실행하지 않아도 돼요.
```python
from google.colab import files # 구글 코랩의 files 모듈 불러오기
irisData1 = files.upload() # files 모듈의 upload()로 파일 업로드하기
irisData2 = files.upload()
```

데이터셋 불러오기

In [ ]:
import pandas as pd # pandas 모듈을 불러와 별칭을 pd로 지정
df1 = pd.read_csv('https://raw.githubusercontent.com/sweet1203/ds_ai_class/main/lessons/u2-preprocess-2/assets/Iris1.csv', encoding = 'cp949') # 데이터프레임으로 변환
df2 = pd.read_csv('https://raw.githubusercontent.com/sweet1203/ds_ai_class/main/lessons/u2-preprocess-2/assets/Iris2.csv', encoding = 'cp949')
df1.head() # df1 앞 5개 데이터 출력

두 개의 데이터 파일 합치기

In [ ]:
df = pd.concat([df1, df2]) # df1과 df2를 위아래 방향으로 합치기
df

결측치 확인하기

In [ ]:
df.isna().sum()

결측치 처리하기

In [ ]:
df3 = df.dropna(axis = 0, inplace = False)
df3

기초 통계량으로 이상치 확인하기

In [ ]:
df3.describe()

In [ ]:
df3.groupby('종류')['꽃받침 길이'].describe()

히스토그램으로 이상치 확인하기

In [ ]:
!pip install koreanize-matplotlib # 한글 폰트 설정 라이브러리 설치

In [ ]:
import koreanize_matplotlib # 한글 출력 가능한 그래프 모듈
import matplotlib.pyplot as plt # 그래프 그리기에 필요한 모듈
import seaborn as sns # 시각화하기 위해 seaborn 모듈 불러오기

In [ ]:
sns.histplot(data = df3, x = '꽃받침 길이', hue = '종류')

In [ ]:
sns.histplot(data = df3, x = '꽃받침 너비', hue = '종류')

In [ ]:
sns.histplot(data = df3, x = '꽃잎 길이', hue = '종류')

In [ ]:
sns.histplot(data = df3, x = '꽃잎 너비', hue = '종류')

박스플롯으로 이상치 확인하기

In [ ]:
sns.catplot(y = '꽃받침 길이', x = '종류', kind = 'box', data = df3, hue ='종류')
sns.catplot(y = '꽃받침 너비', x = '종류', kind = 'box', data = df3, hue = '종류')
sns.catplot(y = '꽃잎 길이', x = '종류', kind = 'box', data = df3, hue = '종류')
sns.catplot(y = '꽃잎 너비', x = '종류', kind = 'box', data = df3, hue = '종류')

이상치 제거하기

(1) 사분위수 범위

In [ ]:
Q1 = df3.groupby('종류')['꽃받침 길이'].quantile(0.25) # 제1사분위수
Q3 = df3.groupby('종류')['꽃받침 길이'].quantile(0.75) # 제3사분위수
IQR = Q3 - Q1 # 사분위수 범위
print('Q1:', Q1); print('Q3:', Q3); print('IQR:', IQR)

(2) 안팎 울타리값

In [ ]:
lower = Q1 - 1.5 * IQR; upper = Q3 + 1.5 * IQR
print('lower:', lower); print('upper:', upper)

(3) 이상치 찾기

In [ ]:
df4 = df3[df3['종류'] == '버시컬러']
df4[(df4['꽃받침 길이'] < lower['버시컬러'] ) | (df4['꽃받침 길이'] > upper['버시컬러'])]

In [ ]:
#이상치 찾기 함수
outList=[]
def outlier(df,pro,tp):
  global outList
#df - 데이터 프레임, pro - 속성, tp - 종류
  q1= df.groupby('종류')[pro].quantile(0.25)
  q3 = df.groupby('종류')[pro].quantile(0.75)
  IQR = q3 - q1
  lower = q1 - 1.5 *IQR
  upper = q3 +1.5 *IQR
  df = df[df['종류']== tp]
  df = df[ (df[pro] < lower[tp] ) | ( df[pro] > upper[tp] ) ]
  outList.append(df.index.to_numpy().tolist()) #데이터프레임의 값을 넘파이로 변경한 후 리스트로 바꾸어  outList에 저장하기

  return df

(4) 이상치 제거하기

In [ ]:
df3 = df3.drop(index = [3, 5, 6, 10, 13, 14, 18, 20, 22, 23, 24, 35, 43, 44,
                        98, 106], axis = 0) # 제거하려는 인덱스 설정

### 더 알아보기
이상치를 제거한 뒤 남은 데이터의 크기와 종류별 개수를 확인해 보자. 몇 행이 남았나요? 종류별 개수는 고르게 남았나요?
(`pd.concat`은 원래 인덱스를 그대로 가져와서 Iris1 부분과 Iris2 부분의 인덱스 번호가 겹쳐요. `drop(index=[...])`은 그 번호를 가진 행을 모두 지워요.)

In [ ]:
print(df3.shape)
df3['종류'].value_counts()

산점도로 속성 간의 관계 알아보기

In [ ]:
sns.pairplot(df3, hue = '종류', markers = ['o','s','D'])
plt.show()

히트맵을 활용하여 속성 간의 관계 알아보기

In [ ]:
irisData = df3.corr(method = 'pearson', numeric_only = True)
plt.figure(figsize = (10, 10))
sns.heatmap(data = irisData, annot = True, fmt = '.2f', linewidth = .5,
cmap = 'RdYlGn_r')
plt.show()